# NovaGen Health Risk Classification — Guided Analysis

This notebook explains the supervised-learning workflow step by step. The original assignment notebook is kept separately in `novagen_original.ipynb`. This expanded notebook is intended to be executed from the repository root.

**Important:** educational project only; not a clinical tool. The assignment does not document which numeric target label means healthy or unhealthy.

## 1. Imports and reproducibility

We use pandas for tabular data, matplotlib for visualization, and scikit-learn for splitting, preprocessing, estimators, and metrics. A fixed random seed makes the hold-out split reproducible.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, ConfusionMatrixDisplay
from src.train import TARGET_COLUMN, RANDOM_STATE, TEST_SIZE, load_dataset, build_models

DATA_PATH = Path('data/novagen_dataset.csv')
OUTPUT_DIR = Path('reports')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Dataset path:', DATA_PATH.resolve())

## 2. Load and validate the data

This calls the reusable data-validation function from `src/train.py`. It checks that the target exists, the target has two classes, features are numeric, and missing values are absent. We do not overwrite the original CSV.

In [ ]:
data = load_dataset(DATA_PATH)
print('Rows, columns:', data.shape)
display(data.head())
display(data.dtypes.to_frame('dtype'))

## 3. Inspect target balance

A classifier needs enough examples from each class. `value_counts` shows the observed label distribution. Since the assignment does not define the meaning of labels 0 and 1, we retain numeric labels rather than inventing a health interpretation.

In [ ]:
target_counts = data[TARGET_COLUMN].value_counts().sort_index()
display(target_counts.rename('count').to_frame())
target_counts.plot(kind='bar', title='Target label distribution', xlabel='Numeric target label', ylabel='Records')
plt.tight_layout()
plt.show()

## 4. Separate predictors from the target

`X` contains all predictor columns and `y` contains only `Target`. Keeping the target out of the predictors prevents target leakage.

In [ ]:
X = data.drop(columns=[TARGET_COLUMN])
y = data[TARGET_COLUMN]
print('Feature matrix:', X.shape)
print('Target:', y.shape)
print('Target accidentally in X?', TARGET_COLUMN in X.columns)

## 5. Train/test split

We hold out 20% for evaluation. `stratify=y` preserves class proportions across partitions. The test partition must not be used while fitting preprocessing or model parameters.

scikit-learn reference: [train_test_split](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html).

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)
print('Train rows:', len(X_train), 'Test rows:', len(X_test))
print('Training label proportions:')
display(y_train.value_counts(normalize=True).sort_index().rename('proportion').to_frame())
print('Test label proportions:')
display(y_test.value_counts(normalize=True).sort_index().rename('proportion').to_frame())

## 6. Why a pipeline and scaler?

`StandardScaler` standardizes features using statistics learned on training data. This is useful for models such as KNN and Logistic Regression. A `Pipeline` fits the scaler and classifier together, preventing the scaler from learning from the held-out test rows. Tree-based models are usually less scale-sensitive, but using a consistent pipeline makes evaluation easy to reproduce.

References: [StandardScaler](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html), [Pipelines](https://scikit-learn.org/stable/modules/compose.html), [data leakage](https://scikit-learn.org/stable/common_pitfalls.html#data-leakage).

## 7. Train and compare five classifiers

- **Logistic Regression:** linear probabilistic classifier.
- **K-Nearest Neighbors:** uses the labels of nearby training samples.
- **Random Forest:** aggregates predictions from multiple decision trees.
- **Gradient Boosting:** adds trees in stages to improve the loss.
- **Soft Voting:** averages class probabilities from multiple classifiers.

The parameter choices are a baseline, not a hyperparameter search. See [linear models](https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression), [nearest neighbors](https://scikit-learn.org/stable/modules/neighbors.html), and [ensemble methods](https://scikit-learn.org/stable/modules/ensemble.html).

In [ ]:
rows = []
fitted_models = {}
for name, model in build_models().items():
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)
    report = classification_report(y_test, predictions, output_dict=True, zero_division=0)
    positive_class = 1 if 1 in set(y.unique()) else sorted(y.unique())[-1]
    rows.append({
        'model': name,
        'accuracy': report['accuracy'],
        'precision_class_1': report[str(positive_class)]['precision'],
        'recall_class_1': report[str(positive_class)]['recall'],
        'f1_class_1': report[str(positive_class)]['f1-score'],
    })
    fitted_models[name] = model

comparison = pd.DataFrame(rows).sort_values('recall_class_1', ascending=False).reset_index(drop=True)
display(comparison.style.format({c: '{:.3f}' for c in comparison.columns if c != 'model'}))

## 8. Understand the metrics

**Accuracy** is the overall fraction of correct predictions. **Precision for label 1** asks how often a prediction of label 1 is correct. **Recall for label 1** asks how many true label-1 examples are recovered. **F1** combines precision and recall. For health-oriented tasks, metric choice should reflect the actual consequences of errors; the target labels must be understood before interpreting the metric clinically.

Reference: [scikit-learn classification metrics](https://scikit-learn.org/stable/modules/model_evaluation.html#classification-metrics).

In [ ]:
best_name = comparison.iloc[0]['model']
print('Highest recall for numeric label 1 on this split:', best_name)
ConfusionMatrixDisplay.from_estimator(fitted_models[best_name], X_test, y_test)
plt.title(f'Confusion matrix: {best_name}')
plt.tight_layout()
plt.show()

## 9. Save the comparison

This notebook writes its comparison table for inspection. The command-line pipeline (`python -m src.train`) produces the full collection of reports, chart, metadata, and a serialized model.

In [ ]:
comparison.to_csv(OUTPUT_DIR / 'notebook_model_comparison.csv', index=False)
print('Saved:', (OUTPUT_DIR / 'notebook_model_comparison.csv').resolve())

## 10. Limitations and next steps

1. Confirm the meaning and coding of the target and every encoded feature.
2. Inspect class-level errors, not only aggregate accuracy.
3. Use cross-validation for more robust comparisons and tune hyperparameters only on training data.
4. Consider fairness, privacy, external validation, and domain-expert review before any health-related deployment.

Cross-validation reference: [scikit-learn model selection](https://scikit-learn.org/stable/modules/cross_validation.html).

**This notebook demonstrates machine-learning practice only. It must not be used to diagnose, treat, screen, or select real patients.**